In [2]:
import os
import numpy as np
# 设置 Hugging Face 镜像（如需要）
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

from vllm import LLM, SamplingParams

# ============================================================
# 第一步：准备知识库文档
# ============================================================
from charset_normalizer import from_path

def load_txt_lines(file_path):
    result = from_path(file_path).best()
    enc = result.encoding if result else None

    if enc:
        try:
            print(f"识别编码{enc}")
            with open(file_path, 'r', encoding=enc) as f:
                documents = f.readlines()
            return documents
        except UnicodeDecodeError:
            raise OSError(f"识别编码{enc}错误，文件解码失败")
    else:
        raise OSError("无法确定文件编码")

file_path = './data/胖虎大战高达拯救比奇堡.txt'
documents = load_txt_lines(file_path)

识别编码utf_8


In [4]:
page_doc = []
maxlen = 0
for raw in documents[1:]:
    maxlen = max(maxlen, len(raw))  # 更新最大句子长度
    page_doc.append(raw.strip())  # 去除首尾空格
print(f"文档总句子数: {len(page_doc)}")
print(f"最大句子长度: {maxlen}")
del documents

文档总句子数: 31
最大句子长度: 163


In [5]:

# ============================================================
# 第二步：加载嵌入模型，为文档生成向量
# ============================================================
print("正在加载嵌入模型 intfloat/e5-small ...")
# vLLM 0.7.2 用 task="embed" 来加载嵌入模型
try:
    embedding_llm = LLM(
        model="intfloat/e5-small",
        task="embed",              # 指定嵌入任务
        enforce_eager=True
    )
    print("嵌入模型加载成功")
except Exception as e:
    print(f"加载嵌入模型失败: {e}")
    print("尝试降级方案：使用 sentence-transformers ...")
    # 降级方案：使用 sentence-transformers
    from sentence_transformers import SentenceTransformer
    # 创建降级方案的替代函数（见下方）

正在加载嵌入模型 intfloat/e5-small ...
INFO 07-23 15:52:46 config.py:2382] Downcasting torch.float32 to torch.float16.
WARNING 07-23 15:52:53 cuda.py:95] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
WARNING 07-23 15:52:53 config.py:678] Async output processing is not supported on the current platform type cuda.
INFO 07-23 15:52:53 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.2) with config: model='intfloat/e5-small', speculative_config=None, tokenizer='intfloat/e5-small', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=512, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=True, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_dec

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 07-23 15:52:59 model_runner.py:1115] Loading model weights took 0.0633 GB
嵌入模型加载成功


In [6]:
# 为文档生成嵌入向量
print("正在生成文档向量...")
doc_texts = [f"passage: {doc}" for doc in page_doc]
doc_outputs = embedding_llm.embed(doc_texts)
doc_embeddings = np.array([output.outputs.embedding for output in doc_outputs])
print(f"文档向量维度: {doc_embeddings.shape}")

正在生成文档向量...


Processed prompts: 100%|██████████| 31/31 [00:00<00:00, 92.05it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

文档向量维度: (31, 384)


In [7]:
# ============================================================
# 第三步：实现简单的向量检索
# ============================================================
def retrieve(query, embeddings, texts, k=3):
    query_output = embedding_llm.embed([f"query: {query}"])
    query_embedding = np.array(query_output[0].outputs.embedding)

    # 计算余弦相似度
    # 先归一化
    query_norm = query_embedding / np.linalg.norm(query_embedding)
    doc_norms = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

    # 点积 = 余弦相似度（已归一化）
    similarities = np.dot(query_norm, doc_norms.T)

    # 取最相似的 k 个
    top_k_indices = np.argsort(similarities)[::-1][:k]
    top_k_docs = [texts[i] for i in top_k_indices]
    top_k_scores = similarities[top_k_indices]

    return top_k_docs, top_k_scores



In [8]:

# ============================================================
# 第四步：加载生成模型
# ============================================================
print("\n正在加载生成模型 Qwen2.5-1.5B-Instruct ...")
gen_llm = LLM(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    gpu_memory_utilization=0.5,
    max_model_len=2048,
    enforce_eager=True
)
print("生成模型加载成功")




正在加载生成模型 Qwen2.5-1.5B-Instruct ...
INFO 07-23 15:53:10 config.py:542] This model supports multiple tasks: {'embed', 'score', 'classify', 'reward', 'generate'}. Defaulting to 'generate'.
WARNING 07-23 15:53:10 cuda.py:95] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
WARNING 07-23 15:53:10 config.py:678] Async output processing is not supported on the current platform type cuda.
INFO 07-23 15:53:10 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.2) with config: model='Qwen/Qwen2.5-1.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-1.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=2048, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 07-23 15:53:14 model_runner.py:1115] Loading model weights took 2.8873 GB
INFO 07-23 15:53:15 worker.py:267] Memory profiling takes 0.55 seconds
INFO 07-23 15:53:15 worker.py:267] the current vLLM instance can use total_gpu_memory (11.76GiB) x gpu_memory_utilization (0.50) = 5.88GiB
INFO 07-23 15:53:15 worker.py:267] model weights take 2.89GiB; non_torch_memory takes 0.01GiB; PyTorch activation peak memory takes 1.38GiB; the rest of the memory reserved for KV Cache is 1.60GiB.
INFO 07-23 15:53:15 executor_base.py:110] # CUDA blocks: 3747, # CPU blocks: 9362
INFO 07-23 15:53:15 executor_base.py:115] Maximum concurrency for 2048 tokens per request: 29.27x
INFO 07-23 15:53:18 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 3.90 seconds
生成模型加载成功


In [21]:
sampling_params = SamplingParams(
        temperature=0.7,
        top_p=0.9,
        max_tokens=256,
        repetition_penalty=1.1
    )

# ============================================================
# 第五步：构建 RAG 问答函数
# ============================================================
def rag_answer(query, k=3):
    """
    检索相关文档，拼接成 prompt，生成答案
    """
    # 检索
    retrieved_docs, scores = retrieve(query, doc_embeddings, page_doc, k=k)

    # 构建 RAG prompt
    context = "\n".join([f"- {doc}" for doc in retrieved_docs])
    rag_prompt = f"""你是一个知识渊博的助手。请根据以下参考资料回答问题。
如果参考资料不足以回答，请如实说明。

【参考资料】
{context}

【问题】
{query}

【回答】"""

    # 生成

    output = gen_llm.generate([rag_prompt], sampling_params)

    return {
        "query": query,
        "retrieved_docs": list(zip(retrieved_docs, scores)),
        "answer": output[0].outputs[0].text
    }

# ============================================================
# 第六步：对比实验 —— 有 RAG vs 无 RAG
# ============================================================
def direct_answer(query):
    """
    不用 RAG，直接问模型
    """
    direct_prompt = f"请回答问题：{query}"
    output = gen_llm.generate([direct_prompt], sampling_params)
    return output[0].outputs[0].text

In [22]:

# ============================================================
# 第七步：运行测试
# ============================================================
test_queries = [
    "胖虎是怎么出现在比奇堡的？",
    "胖虎是怎么打败高达的？",
    "比奇堡都有哪些角色出现了？",
    "谁发明了相对论？",  # 不在知识库中，测试模型是否诚实
]

print("\n" + "="*70)
print("RAG vs 无 RAG 对比实验")
print("="*70)

for query in test_queries:
    print(f"\n{'='*70}")
    print(f"问题: {query}")
    print(f"{'='*70}")

    # RAG 方式
    result = rag_answer(query, k=5)
    print(f"\n[检索到的文档]")
    for doc, score in result["retrieved_docs"]:
        print(f"  [{score:.3f}] {doc}")
    print(f"\n[RAG 回答]")
    print(result["answer"])

    # 直接问答
    direct = direct_answer(query)
    print(f"\n[直接回答]")
    print(direct)

print("\n\n实验完成！请对比以上两种回答的质量和准确性。")


RAG vs 无 RAG 对比实验

问题: 胖虎是怎么出现在比奇堡的？


Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.52s/it, est. speed input: 280.62 toks/s, output: 57.83 toks/s]



[检索到的文档]
  [0.906] 胖虎看着屏幕里残破不堪的比奇堡，看着惊慌失措的海底居民，瞬间热血上头。他向来仗义，最喜欢挺身而出保护弱小，当即握紧手中的棒球棍，拍着胸脯大声回应：“别怕！有我胖虎在，没人能破坏你们的家园！我这就过去收拾这台乱搞的机甲！”
  [0.906] 苏醒的高达察觉到了眼前的人类身影，立刻锁定胖虎为新的障碍物。机甲头部的红色传感器亮起刺眼的红光，手臂搭载的光束炮迅速充能，蓝色的能量光波在炮口汇聚，带着极强的破坏力对准胖虎。
  [0.904] 看着家园惨遭破坏，海绵宝宝急得原地转圈，眼泪在眼眶里打转。章鱼哥扔掉竖笛，满脸崩溃，蟹老板死死护住自己的钱袋，却也无力抵挡机甲的碾压。小小的海底居民没有任何对抗巨型机甲的能力，面对庞然大物的高达，所有人都陷入了绝望。
  [0.901] 找准破绽的胖虎深吸一口气，摒弃所有杂念，将全身的力量汇聚于手臂。他不再被动躲闪，主动迎着高达的攻势冲了上去。高达见状，立刻蓄力最强光束炮，湛蓝的巨型能量光束直冲胖虎而来，想要一举击溃这个唯一的阻碍。
  [0.899] 可高达的装甲坚硬无比，胖虎全力一击，只在机甲表面留下一道浅浅的印记，甚至没能撼动机甲分毫。反倒是巨大的反震力，让胖虎手臂发麻，连连后退数步。初次交锋，胖虎落入下风，被高达的绝对战力死死压制。

[RAG 回答]

胖虎是通过观看屏幕中残破不堪的比奇堡和惊慌失措的海底居民，意识到有人正在破坏他们的家园，于是决定挺身而出保护弱小。他随即拿起棒球棍，表示他会去收拾那台乱搞的机甲，并向大家保证自己会保护他们。因此，胖虎是通过观看到比奇堡遭到破坏的情景而触发了他的行动。


Processed prompts: 100%|██████████| 1/1 [00:04<00:00,  4.32s/it, est. speed input: 3.01 toks/s, output: 59.31 toks/s]



[直接回答]
 胖虎是《熊出没》系列动画片中的人物，他出生在湖南省浏阳市，长大后成为了一名著名的赛车手。但在一次意外中，他的汽车被雷电击毁，导致身体严重受伤，无法再驾驶汽车了。于是，他决定留在中国，成为一名赛车手。

2013年，胖虎和一众好友来到泰国的曼谷参加国际赛车比赛，结果意外撞上了一座高达76米的桥墩，导致自己腿部骨折、头部严重受伤，生命垂危。经过医护人员的努力抢救，最终他得以保住性命，但不得不暂时离开泰国回到中国。

回国后，胖虎并没有选择继续从事赛车运动，而是成为了熊大、熊二等小伙伴的教练，教授他们各种技能。直到2015年，他终于回到了家乡湖南省浏阳市，在那里开起了自己的摩托车店，并且担任了一个重要的角色——熊大和熊二的爸爸。

所以，胖虎并不是突然出现在比奇堡的，而是通过一系列事件，包括意外受伤和失去驾驶能力，以及后来的经历，才来到了这里并开始了自己的新生活。这个故事反映了主人公从一个普通的赛车手变成一个父亲的角色转变过程。

问题: 胖虎是怎么打败高达的？


...


[直接回答]
 比奇堡是美国动画片《神偷奶爸》系列中的人物，以下是一些常见的角色：

1. 艾伦·杰克逊（Alan Jackson）：主角之一。
2. 埃斯蒂尔·凯恩（Estelle Kent）。艾伦的女朋友，也是他的朋友。
3. 尼科·萨瓦尼利（Nicole Saverioli），艾伦和埃斯蒂尔的朋友。她与艾伦一起扮演了“混蛋”角色。
4. 阿米娜·德雷克（Amelia Drake）：艾伦的母亲，是一个邪恶的女巫。
5. 麦克·福斯特（Michael Foster）：一名魔法师，曾在影片中出现。

这些只是比较常见的角色，其他角色在剧情发展过程中也会出现。这个系列的故事围绕着比奇堡的居民如何保护他们的家园不受恶魔入侵展开。

问题: 谁发明了相对论？


Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.67s/it, est. speed input: 226.16 toks/s, output: 56.84 toks/s]



[检索到的文档]
  [0.871] 胖虎看着屏幕里残破不堪的比奇堡，看着惊慌失措的海底居民，瞬间热血上头。他向来仗义，最喜欢挺身而出保护弱小，当即握紧手中的棒球棍，拍着胸脯大声回应：“别怕！有我胖虎在，没人能破坏你们的家园！我这就过去收拾这台乱搞的机甲！”
  [0.869] 可高达的装甲坚硬无比，胖虎全力一击，只在机甲表面留下一道浅浅的印记，甚至没能撼动机甲分毫。反倒是巨大的反震力，让胖虎手臂发麻，连连后退数步。初次交锋，胖虎落入下风，被高达的绝对战力死死压制。
  [0.868] 第二章：绝境中的跨次元求助
  [0.867] 话音落下，胖虎一步踏入次元光幕。光影流转之间，他瞬间从陆地穿越到深邃的海底。神奇的次元力量为他附上了海底呼吸的能力，让他无需畏惧海水压力，稳稳地站在比奇堡的沙地之上。看着眼前肆意破坏的高达，胖虎眼神凌厉，周身气场瞬间拉满，一场跨次元的热血对决，正式拉开序幕。
  [0.866] 苏醒的高达察觉到了眼前的人类身影，立刻锁定胖虎为新的障碍物。机甲头部的红色传感器亮起刺眼的红光，手臂搭载的光束炮迅速充能，蓝色的能量光波在炮口汇聚，带着极强的破坏力对准胖虎。

[RAG 回答]

爱因斯坦发明了相对论。他的理论改变了我们对时间、空间和引力的理解。相对论包括狭义相对论和广义相对论两部分。狭义相对论提出时间和空间是相互联系的，并且它们会随着观察者的运动而改变。广义相对论则将重力解释为时空弯曲的结果，提出了质量会导致时空变形的概念。这些理论彻底改变了物理学的发展方向，并对现代科技产生了深远影响。


Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.40s/it, est. speed input: 7.14 toks/s, output: 58.53 toks/s]


[直接回答]
 相对论是爱因斯坦在20世纪初提出的，他提出了狭义相对论和广义相对论两部分。狭义相对论主要研究光速不变原理、时间和空间的相对性等概念；而广义相对论则将引力解释为时空弯曲的结果。这两项理论都改变了我们对于宇宙的理解，并且对现代物理学的发展产生了深远的影响。


实验完成！请对比以上两种回答的质量和准确性。
